# Colonoscopy Data Preprocessing

This notebook preprocesses colonoscopy lesion data for feature selection analysis.

**Dataset**: Gastrointestinal Lesions in Regular Colonoscopy  
**Source**: UCI Machine Learning Repository  
**Task**: Binary classification - benign (hyperplasic) vs. malignant (adenoma + serrated adenoma)  
**Samples**: 152 (76 lesions × 2 light types)  
**Features**: 698 (texture, color, and shape features)

## 1. Import Necessary Tools

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
from IPython.display import display, Markdown

# Set display options
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)
pd.set_option("display.precision", 3)

display(Markdown("✓ Libraries imported successfully!"))

✓ Libraries imported successfully!

## 2. Choose File Path

In [2]:
# Define file path
data_dir = Path("../data/colonoscopy")
file_path = data_dir / "data.txt"

display(Markdown(f"**Data directory:** `{data_dir}`"))
display(Markdown(f"**File path:** `{file_path}`"))
display(Markdown(f"**File exists:** {file_path.exists()}"))

**Data directory:** `..\data\colonoscopy`

**File path:** `..\data\colonoscopy\data.txt`

**File exists:** True

## 3. Load CSV Data

The data structure:
- **Row 1**: Sample names (lesion IDs)
- **Row 2**: Lesion type (3=adenoma, 1=hyperplasic, 2=serrated)
- **Row 3**: Light type (1=WL white light, 2=NBI narrow band imaging)
- **Row 4+**: 698 features (texture, color, shape)

In [3]:
# Load the data (no header, comma-separated)
data_raw = pd.read_csv(file_path, header=None)

display(Markdown(f"**Raw data shape:** {data_raw.shape}"))
display(
    Markdown(
        f"**Rows:** {data_raw.shape[0]} (3 metadata rows + {data_raw.shape[0] - 3} feature rows)"
    )
)
display(Markdown(f"**Columns:** {data_raw.shape[1]} samples"))

**Raw data shape:** (701, 152)

**Rows:** 701 (3 metadata rows + 698 feature rows)

**Columns:** 152 samples

In [4]:
# Extract metadata rows
sample_names = data_raw.iloc[0].values  # Row 1: sample names
lesion_types = data_raw.iloc[1].astype(int).values  # Row 2: lesion type
light_types = data_raw.iloc[2].astype(int).values  # Row 3: light type

# Extract feature data (rows 4+)
features = data_raw.iloc[3:].T  # Transpose so samples are rows
features.columns = [f"feature_{i}" for i in range(features.shape[1])]
features.index = sample_names
features.index.name = "sample_name"

display(
    Markdown(
        f"**Features extracted:** {features.shape[0]} samples × {features.shape[1]} features"
    )
)

**Features extracted:** 152 samples × 698 features

In [ ]:
# Create binary response variable
# 1 = hyperplasic (benign) → 0
# 2 = serrated adenoma (malignant) → 1
# 3 = adenoma (malignant) → 1

response = np.where(lesion_types == 1, 0, 1)  # 0 = benign, 1 = malignant

# Create complete dataframe
df = features.copy()
df["lesion_type"] = lesion_types
df["light_type"] = light_types
df["response"] = response

display(Markdown("✓ Dataset created successfully!"))
display(Markdown(f"**Shape:** {df.shape[0]} samples × {df.shape[1]} columns"))
display(
    Markdown(f"**Memory usage:** {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
)

✓ Dataset created successfully!

**Shape:** 152 samples × 701 columns

**Memory usage:** 5.47 MB

## 4. Display Dataset Overview

In [6]:
# Display first few rows
display(Markdown("**First 5 rows:**"))
display(df.head())

**First 5 rows:**

,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,feature_10,feature_11,feature_12,feature_13,feature_14,feature_15,feature_16,feature_17,feature_18,feature_19,feature_20,feature_21,feature_22,feature_23,feature_24,...,feature_676,feature_677,feature_678,feature_679,feature_680,feature_681,feature_682,feature_683,feature_684,feature_685,feature_686,feature_687,feature_688,feature_689,feature_690,feature_691,feature_692,feature_693,feature_694,feature_695,feature_696,feature_697,lesion_type,light_type,response
sample_name,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
adenoma_1,138.12,1606.8,0.0038753,0.0058799,0.0052134,0.0069351,0.0073333,0.0095801,0.0073796,0.0067515,0.0081109,0.0078631,0.01403,0.021471,0.0073058,0.0065843,0.0065763,0.0067911,0.0063111,0.0078946,0.0037612,0.0036387,0.003145,0.0035961,0.0069674,...,0.019825,0.019445,0.019124,0.018844,0.018476,0.018043,0.017338,0.016502,0.016154,0.015642,0.014721,0.014486,0.013994,0.013532,0.013157,0.012743,0.012613,0.012422,0.012252,0.011377,0.011198,0.011131,3,1,1
adenoma_1,127.99,3377.9,0.0035637,0.0056453,0.0058914,0.0082193,0.0073251,0.0098827,0.0072881,0.0066991,0.0088768,0.0090356,0.013744,0.020926,0.0070707,0.0065929,0.0062685,0.0064428,0.0056026,0.006596,0.0033124,0.0032993,0.0026269,0.002919,0.0068612,...,0.019825,0.019445,0.019124,0.018844,0.018476,0.018043,0.017338,0.016502,0.016154,0.015642,0.014721,0.014486,0.013994,0.013532,0.013157,0.012743,0.012613,0.012422,0.012252,0.011377,0.011198,0.011131,3,2,1
adenoma_8,80.415,1852.1,0.0047612,0.0075581,0.0052945,0.0068631,0.0060879,0.0070738,0.008361,0.0092981,0.01099,0.011964,0.018085,0.024475,0.011661,0.011492,0.01177,0.013024,0.012922,0.01599,0.0090393,0.0093796,0.0071518,0.0081998,0.0072285,...,0.005916,0.00548,0.005411,0.005099,0.004977,0.004739,0.0045,0.004413,0.004146,0.003988,0.003885,0.003712,0.003564,0.00338,0.003232,0.0032,0.003006,0.002985,0.002922,0.002631,0.00261,0.002531,3,2,1
adenoma_8,90.896,1904.3,0.004147,0.0067283,0.0050606,0.0068787,0.0079478,0.009525,0.010492,0.011988,0.013539,0.015285,0.02032,0.023581,0.014712,0.015039,0.015051,0.01689,0.014944,0.01743,0.0099627,0.0097883,0.0067193,0.0075594,0.0063697,...,0.005916,0.00548,0.005411,0.005099,0.004977,0.004739,0.0045,0.004413,0.004146,0.003988,0.003885,0.003712,0.003564,0.00338,0.003232,0.0032,0.003006,0.002985,0.002922,0.002631,0.00261,0.002531,3,1,1
adenoma_9,106.16,1184.4,0.0055184,0.0076452,0.0079176,0.0095128,0.010927,0.012717,0.011937,0.012203,0.01286,0.012835,0.017757,0.021636,0.011637,0.011371,0.010078,0.010264,0.0091978,0.010632,0.0059571,0.0064241,0.0050621,0.0063657,0.010596,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3,2,1


In [7]:
# Dataset information
display(Markdown("**Dataset Info:**"))
df.info()

**Dataset Info:**

<class 'pandas.core.frame.DataFrame'>
Index: 152 entries, adenoma_1 to serrated_9
Columns: 701 entries, feature_0 to response
dtypes: int64(3), object(698)
memory usage: 833.6+ KB


In [8]:
# Statistical summary
display(Markdown("**Statistical Summary (first 10 features):**"))
display(df.iloc[:, :10].describe())

**Statistical Summary (first 10 features):**

,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9
count,152,152,152,152,152,152,152,152,152,152
unique,152,151,152,152,152,152,152,152,152,152
top,138.12,2240.5,0.0038753,0.0058799,0.0052134,0.0069351,0.0073333,0.0095801,0.0073796,0.0067515
freq,1,2,1,1,1,1,1,1,1,1


In [9]:
# Check for missing values
missing_count = df.isnull().sum().sum()
display(Markdown(f"**Missing values:** {missing_count}"))

if missing_count > 0:
    display(Markdown("**Columns with missing values:**"))
    missing_cols = df.isnull().sum()[df.isnull().sum() > 0]
    display(missing_cols)

**Missing values:** 0

In [10]:
# Lesion type distribution
display(Markdown("**Lesion type distribution:**"))
lesion_labels = {
    1: "Hyperplasic (benign)",
    2: "Serrated adenoma (malignant)",
    3: "Adenoma (malignant)",
}
lesion_counts = pd.Series(lesion_types).value_counts().sort_index()
for lesion_type, count in lesion_counts.items():
    display(
        Markdown(f"- **{lesion_type}** - {lesion_labels[lesion_type]}: {count} samples")
    )

**Lesion type distribution:**

- **1** - Hyperplasic (benign): 42 samples

- **2** - Serrated adenoma (malignant): 30 samples

- **3** - Adenoma (malignant): 80 samples

In [11]:
# Light type distribution
display(Markdown("**Light type distribution:**"))
light_labels = {1: "WL (White Light)", 2: "NBI (Narrow Band Imaging)"}
light_counts = pd.Series(light_types).value_counts().sort_index()
for light_type, count in light_counts.items():
    display(
        Markdown(f"- **{light_type}** - {light_labels[light_type]}: {count} samples")
    )

**Light type distribution:**

- **1** - WL (White Light): 76 samples

- **2** - NBI (Narrow Band Imaging): 76 samples

In [12]:
# Binary response variable distribution
display(Markdown("**Binary response variable (malignant vs. benign):**"))
display(df["response"].value_counts())
display(
    Markdown(
        f"\n**Class balance:** {df['response'].value_counts(normalize=True).round(3).to_dict()}"
    )
)
display(Markdown("- **0** = Benign (hyperplasic)"))
display(Markdown("- **1** = Malignant (adenoma + serrated adenoma)"))

**Binary response variable (malignant vs. benign):**

response
1    110
0     42
Name: count, dtype: int64


**Class balance:** {1: 0.724, 0: 0.276}

- **0** = Benign (hyperplasic)

- **1** = Malignant (adenoma + serrated adenoma)

In [13]:
# Visualize class distribution with Plotly
# Create subplots
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Lesion Type Distribution", "Binary Classification Distribution"),
    specs=[[{"type": "bar"}, {"type": "bar"}]],
)

# Plot 1: Lesion types
lesion_labels = {
    1: "Hyperplasic (benign)",
    2: "Serrated adenoma (malignant)",
    3: "Adenoma (malignant)",
}
lesion_counts = pd.Series(lesion_types).value_counts().sort_index()
lesion_names = [lesion_labels[i] for i in lesion_counts.index]
colors_lesion = ["#2ecc71", "#e67e22", "#e74c3c"]

fig.add_trace(
    go.Bar(
        x=lesion_names,
        y=lesion_counts.values,
        marker_color=colors_lesion,
        name="Lesion Type",
    ),
    row=1,
    col=1,
)

# Plot 2: Binary classification
response_labels = {0: "Benign", 1: "Malignant"}
response_counts = df["response"].value_counts().sort_index()
response_names = [response_labels[i] for i in response_counts.index]
colors_binary = ["#2ecc71", "#e74c3c"]

fig.add_trace(
    go.Bar(
        x=response_names,
        y=response_counts.values,
        marker_color=colors_binary,
        name="Binary Class",
    ),
    row=1,
    col=2,
)

# Update layout
fig.update_xaxes(title_text="Lesion Type", row=1, col=1)
fig.update_xaxes(title_text="Class", row=1, col=2)
fig.update_yaxes(title_text="Count", row=1, col=1)
fig.update_yaxes(title_text="Count", row=1, col=2)
fig.update_layout(
    height=400, showlegend=False, title_text="Class Distribution Analysis"
)

fig.show()

In [22]:
# combine lesion_type and light_type into a single stratification group column
df["stratification_groups"] = (
    df["lesion_type"].astype(str) + "_" + df["light_type"].astype(str)
)
# then drop the original columns to avoid data leakage
df.drop(columns=["lesion_type", "light_type"], inplace=True)

In [23]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 152 entries, adenoma_1 to serrated_9
Columns: 700 entries, feature_0 to stratification_groups
dtypes: int64(1), object(699)
memory usage: 836.5+ KB


In [25]:
# save the preprocessed dataset to csv
data_dir = Path("../data/colonoscopy")
output_file_path = (
    data_dir
    / f"colonoscopy_preprocessed_n={df.shape[0]}_p={df.shape[1] - 2}_response=[malignant_vs_benign]_onlyidentified=False_stratified=True.csv"
)
df.to_csv(output_file_path)
display(Markdown(f"✓ Preprocessed dataset saved to `{output_file_path}`"))

✓ Preprocessed dataset saved to `..\data\colonoscopy\colonoscopy_preprocessed_n=152_p=698_response=[malignant_vs_benign]_onlyidentified=False_stratified=True.csv`